## Потолок ретривера — идеальный (oracle) реранкер

Тот же набор кандидатов, что и без реранкера (`use_rerank=False`), но переранжированный
идеально: все золотые документы, попавшие в выдачу ретривера, поднимаются наверх.
Это верхняя граница того, что вообще можно выбить на нашем датасете при текущем ретривере —
золотые документы, которые ретривер не вернул, никакой реранкер уже не восстановит.

In [1]:
import polars as pl

In [2]:
golden_set = pl.read_parquet("../data/golden_set.parquet").filter(pl.col("geonameIds").list.len() > 0)

In [3]:
import httpx
from tqdm.auto import tqdm


def search_batch(queries: list[str], top_k: int = 50, use_rerank: bool = True):
    base_url = "http://localhost:8000/v1/search"
    results = []
    
    with httpx.Client(timeout=30.0) as client:  # синхронный клиент
        for query in tqdm(queries):
            response = client.get(base_url, params={"text": query, "top_k": top_k, "use_rerank": use_rerank})
            response.raise_for_status()
            results.append(response.json())
    
    return results

In [4]:
from ir_measures import P, Recall, RR, calc


qrels_dict = {}
for row in golden_set.iter_rows(named=True):
    qrels_dict.update({row["query"]: {str(gid): 1 for gid in row["geonameIds"]}})

predictions = search_batch(qrels_dict.keys(), top_k=50, use_rerank=False)

  0%|          | 0/180 [00:00<?, ?it/s]

In [5]:
# Идеальный (oracle) реранкер: среди кандидатов, которые вернул ретривер,
# поднимаем все золотые документы наверх (score=1), остальные — вниз (score=0).
for pred in predictions:
    gold = {int(g) for g in qrels_dict[pred["query"]]}
    for r in pred["results"]:
        r["score"] = 1.0 if r["geonameid"] in gold else 0.0

In [6]:
run_dict = {}
for pred in predictions:
    run_dict.update({pred["query"]: {str(r["geonameid"]): r["score"] for r in pred["results"]}})

metrics = calc([RR, P@1, Recall@5, Recall@10, Recall@25, Recall@50], qrels_dict, run_dict)
metrics_aggregated = pl.DataFrame({str(k): v for k, v in metrics.aggregated.items()}).unpivot().sort("value")

metrics_per_query = {str(m): {"query": [], "value": []} for m in metrics.aggregated}

for metric in metrics.per_query:
    mname = str(metric.measure)
    metrics_per_query[mname]["query"].append(metric.query_id)
    metrics_per_query[mname]["value"].append(metric.value)

for mname in metrics_per_query:
    metrics_per_query[mname] = pl.DataFrame(metrics_per_query[mname]).sort("value")

In [7]:
metrics_aggregated

variable,value
str,f64
"""R@5""",0.937839
"""R@10""",0.949839
"""R@25""",0.960308
"""R@50""",0.961524
"""RR""",0.988889
"""P@1""",0.988889


In [8]:
for k in metrics_per_query:
    print(k)
    display(metrics_per_query[k].limit(5))

RR


query,value
str,f64
"""Подскажите, какую музыку можно…",0.0
"""Ağrı Dağı'na tırmanmak için en…",0.0
"""Какой самый дешевый способ доб…",1.0
"""Kazan'da toplu taşıma nasıl? M…",1.0
"""What are the best things to do…",1.0


R@10


query,value
str,f64
"""Подскажите, какую музыку можно…",0.0
"""Ağrı Dağı'na tırmanmak için en…",0.0
"""Сколько лететь из Портленда в …",0.3125
"""What are the best weekend geta…",0.434783
"""Is Omsk or Yaroslavl a better …",0.5


R@5


...

query,value
str,f64
"""Подскажите, какую музыку можно…",0.0
"""Ağrı Dağı'na tırmanmak için en…",0.0
"""Is Omsk or Yaroslavl a better …",0.5
"""Van, Sinop ve Şanlıurfa’dan bi…",0.5
"""Какая погода в Ване в конце ок…",0.5


R@50


query,value
str,f64
"""Подскажите, какую музыку можно…",0.0
"""Ağrı Dağı'na tırmanmak için en…",0.0
"""Is Omsk or Yaroslavl a better …",0.5
"""Van, Sinop ve Şanlıurfa’dan bi…",0.5
"""Какая погода в Ване в конце ок…",0.5


P@1


query,value
str,f64
"""Подскажите, какую музыку можно…",0.0
"""Ağrı Dağı'na tırmanmak için en…",0.0
"""Какой самый дешевый способ доб…",1.0
"""Kazan'da toplu taşıma nasıl? M…",1.0
"""What are the best things to do…",1.0
